# Três minissistemas de IA para redes elétricas inteligentes

Sistema especialista, planejamento automático (STRIPS/GPS) e busca A\*, sobre um
**cenário sintético de 60 nós** com dois acessos independentes.

Este caderno acompanha a apresentação. Ele já vem **executado**: as saídas abaixo
foram produzidas pelo código deste repositório, não transcritas. Para reexecutar,
use *Run All* — não é preciso rede nem simulador.

> **Limite de validade.** A topologia é um MODELO sintético. Os custos são
> estimativas de tempo de transporte em milissegundos, não latência medida, e os
> limiares das regras são nominais e não calibrados.

In [1]:
# Works both in a local clone and in Colab, where the notebook opens on its own.
import sys, pathlib, itertools, math, subprocess

REPO = "https://github.com/fsd-dantas/ai4systems.git"
local = pathlib.Path.cwd().parent / "src"
if local.is_dir():
    sys.path.insert(0, str(local))                      # clone: use the checkout
else:                                                    # Colab: fetch it once
    if not pathlib.Path("ai4systems").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    sys.path.insert(0, str(pathlib.Path("ai4systems") / "topics" / "symbolic-network-restoration" / "src"))

from aisg.domain import load_topology
from aisg.expert_system import InferenceEngine, SIM_CASES, build_simulated_knowledge_base
from aisg.planning import (build_restoration_problem, plan_with_astar, plan_with_gps,
                           problem_from_diagnosis)
from aisg.planning.planning_graph import analyse
from aisg.planning.strips import Predicate
from aisg.search import RoutingProblem
from aisg.search.algorithms import (astar, breadth_first, depth_first,
                                    greedy_best_first, uniform_cost)

LANG = "pt"
topology = load_topology("dual")
print(f"{len(topology.nodes)} nos, {len(list(topology.active_links()))} enlaces")

60 nos, 74 enlaces


---
## 1. Sistema especialista

Base **simulada**: 41 regras em cinco camadas, com fatores de certeza no estilo
MYCIN. As regras de diagnóstico nunca leem a medição bruta — leem a conclusão da
camada anterior, de modo que trocar um sensor altera uma camada e nada mais.

In [2]:
kb = build_simulated_knowledge_base()
askable = [v for v in kb.variables.values() if v.askable]
print(f"regras: {len(kb.rules)}   variaveis: {len(kb.variables)}   perguntaveis: {len(askable)}")
for rule_id in ("S01", "S10", "S25", "S35"):
    rule = next(r for r in kb.rules if r.id == rule_id)
    print(f"\n{rule.id}  CF {rule.cf:+.2f}")
    print("   ", rule.rationale_pt)

regras: 41   variaveis: 18   perguntaveis: 13

S01  CF +0.90
    Potencia e relacao sinal-ruido abaixo do limiar.

S10  CF +0.85
    Sinal degradado com emissor ativo no mesmo canal.

S25  CF -0.80
    Retransmissao baixa e evidencia CONTRA contencao de MAC.

S35  CF +1.00
    Alterar o cenario durante uma campanha exige janela autorizada.


### 1.1 Encadeamento progressivo e explicação

O motor responde **por que** pergunta algo durante a consulta e **como** chegou a
uma conclusão depois dela, percorrendo as regras de apoio até os fatos fornecidos.

In [3]:
def diagnose(case_name):
    engine = InferenceEngine(build_simulated_knowledge_base())
    for variable, value in SIM_CASES[case_name].items():
        engine.given(variable, value)
    return engine.forward_chain()

consultation = diagnose("mac_contention")
for goal, facts in consultation.conclusions().items():
    for fact in facts[:2]:
        print(f"{goal:<22} {fact.value:<28} CF {fact.cf:+.3f}")

diagnosis              mac_contention               CF +0.941
recommended_action     separate_channels            CF +0.800
authorization_required yes                          CF +0.800


In [4]:
print(consultation.how("diagnosis", "mac_contention", LANG))

diagnostico = mac_contention (CF +0.94, S14+S15)
  <= S14: SE taxa de retransmissao MAC > 30.0 E qualidade do sinal != poor E emissor no mesmo canal = no ENTAO diagnostico = mac_contention (CF +0.85)
     Retransmissao alta com sinal saudavel: o meio esta disputado, nao fraco.
    taxa de retransmissao MAC = 62.0 % (CF +1.00, given) — fato inicial
    qualidade do sinal = good (CF +0.90, S04)
      <= S04: SE potencia recebida >= -85.0 E relacao sinal-ruido >= 15.0 ENTAO qualidade do sinal = good (CF +0.90)
         Potencia e relacao sinal-ruido dentro do esperado.
        potencia recebida = -78.0 dBm (CF +1.00, given) — fato inicial
        relacao sinal-ruido = 22.0 dB (CF +1.00, given) — fato inicial
    emissor no mesmo canal = no (CF +1.00, given) — fato inicial
  <= S15: SE taxa de retransmissao MAC > 30.0 E nos no mesmo canal = many ENTAO diagnostico = mac_contention (CF +0.75)
     Muitos nos no mesmo canal com retransmissao alta: contencao de CSMA-CA.
    taxa de retransmiss

### 1.2 Os oito casos ilustrativos

Cada caso declara, no bloco `INDUCIBLE_BY`, **como seria comandado** no cenário.
Uma condição comandada é repetível e rotulável; uma condição observada em campo é
apenas provável. Essa é a razão metodológica de trabalhar sobre um modelo.

In [5]:
print(f"{'caso':<26}{'diagnostico':<28}{'CF':>7}   {'acao recomendada':<26}")
print("-" * 94)
for name in SIM_CASES:
    conclusions = diagnose(name).conclusions()
    diagnosis = conclusions["diagnosis"][0]
    action = conclusions.get("recommended_action", [None])[0]
    # the case key IS the expected diagnosis, so this column is a check
    ok = "ok" if diagnosis.value == name else "XX"
    print(f"{name:<26}{diagnosis.value:<28}{diagnosis.cf:>+7.3f}   "
          f"{action.value if action else '-':<26}{ok}")

caso                      diagnostico                      CF   acao recomendada          
----------------------------------------------------------------------------------------------
rf_interference           rf_interference              +0.929   change_channel            ok
excess_path_loss          excess_path_loss             +0.924   restore_path_budget       ok
mac_contention            mac_contention               +0.941   separate_channels         ok
node_failure              node_failure                 +0.800   restart_node              ok
upstream_relay_failure    upstream_relay_failure       +0.975   restore_upstream_relay    ok
routing_misconfiguration  routing_misconfiguration     +0.900   fix_routing               ok
congestion                congestion                   +0.888   reroute_traffic           ok
healthy                   healthy                      +0.810   no_action                 ok


---
## 2. Planejamento automático

O sistema especialista termina com um diagnóstico. Ele **não** diz em que ordem
agir. STRIPS representa cada ação por precondições, lista de adição e lista de
remoção; o estado é o conjunto de literais verdadeiros.

In [6]:
problem = problem_from_diagnosis("mac_contention", "ER_03", topology=topology, simulated=True)
plan, _ = plan_with_astar(problem)
print(plan.render(LANG))
valid, why = plan.validate()
print(f"\nvalidacao: {valid}" + (f" ({why})" if why else ""))

 1. request_authorization(ER_03)   [1]
     Solicitar autorizacao e janela de manutencao para ER_03.
 2. stop_run(ER_03)   [2]
     Interromper a execucao de ER_03 para alterar parametros do cenario.
 3. separate_channels(ER_03)   [3]
     Separar os setores em frequencia no cenario de ER_03.
 4. start_run(ER_03)   [2]
     Retomar a execucao de ER_03.
 5. verify_link(ER_03)   [1]
     Verificar o enlace de ER_03 apos a correcao.
 6. record_logbook(ER_03)   [1]
     Registrar a intervencao em ER_03 no livro de registro.
 7. close_work_order(ER_03)   [1]
     Encerrar a ordem de servico de ER_03.
    custo total: 11

validacao: True


### 2.1 GPS e A\* sobre a mesma representação

Duas estratégias, uma representação. O GPS explica o raciocínio; o A\* garante o
custo mínimo. Neste domínio os dois coincidem — e a seção 2.3 mostra por quê.

In [7]:
print(f"{'diagnostico':<28}{'GPS':>6}{'A*':>8}")
print("-" * 44)
for diagnosis in ("mac_contention", "excess_path_loss", "node_failure",
                  "routing_misconfiguration"):
    p = problem_from_diagnosis(diagnosis, "ER_03", topology=topology, simulated=True)
    g, _ = plan_with_gps(p, lang=LANG)
    a, _ = plan_with_astar(p)
    print(f"{diagnosis:<28}{g.cost:>6g}{a.cost:>8g}")

diagnostico                    GPS      A*
--------------------------------------------
mac_contention                  11      11
excess_path_loss                 9       9
node_failure                     5       5
routing_misconfiguration         5       5


### 2.2 Falhas simultâneas

Um literal `cleared-<falha>` **por falha**. A versão anterior usava um único
literal compartilhado, e clarear uma falha bastava para o plano declarar serviço
restaurado com a outra ainda ativa. O defeito foi relatado a partir do uso.

In [8]:
two = build_restoration_problem("ER_03", ["mac-contention", "excess-path-loss"],
                                simulated=True)
plan2, _ = plan_with_astar(two)
print(plan2.render(LANG))
names = [a.operator.name for a in plan2.actions]
print(f"\nstop_run aparece {names.count('stop_run')}x: as correcoes sao agrupadas "
      f"numa unica parada")

 1. request_authorization(ER_03)   [1]
     Solicitar autorizacao e janela de manutencao para ER_03.
 2. stop_run(ER_03)   [2]
     Interromper a execucao de ER_03 para alterar parametros do cenario.
 3. restore_path_budget(ER_03)   [1]
     Restaurar o orcamento de percurso de ER_03 no cenario.
 4. separate_channels(ER_03)   [3]
     Separar os setores em frequencia no cenario de ER_03.
 5. start_run(ER_03)   [2]
     Retomar a execucao de ER_03.
 6. verify_link(ER_03)   [1]
     Verificar o enlace de ER_03 apos a correcao.
 7. record_logbook(ER_03)   [1]
     Registrar a intervencao em ER_03 no livro de registro.
 8. close_work_order(ER_03)   [1]
     Encerrar a ordem de servico de ER_03.
    custo total: 12

stop_run aparece 1x: as correcoes sao agrupadas numa unica parada


### 2.3 Validação do domínio pelo grafo de planejamento

O grafo de planejamento (Blum & Furst, 1997) é usado aqui como **instrumento de
análise**, não como um quarto planejador: construímos o grafo e as relações de
exclusão mútua, mas não a extração de solução.

In [9]:
result = analyse(two, success=Predicate("service-restored", ("ER_03",)),
                 fault_literals=[Predicate("mac-contention", ("ER_03",)),
                                 Predicate("excess-path-loss", ("ER_03",))])
print(result.render(LANG))

Analise do grafo de planejamento: restore-service-ER_03
-------------------------------------------------------
ponto fixo no nivel: 7
limite inferior do plano: 6
operadores mortos: change_channel, dispatch_crew, fix_routing, fix_vlan, monitor_and_wait, realign_antenna, replace_power_unit, reroute_traffic, restart_node, restore_relay
pontos de escolha: nenhum
violacoes do invariante de sucesso: nenhuma


**Nenhum ponto de escolha** é a explicação *estrutural* de por que o GPS nunca
perde para o A\* neste domínio: a análise meios-fins não tem escolha para errar.
Abaixo, o defeito reencontrado como uma exclusão mútua **ausente**.

In [10]:
from aisg.planning.strips import Operator, Problem, make_state

def two_fault(shared):
    cleared = (lambda f: "fault-cleared(?n)") if shared else (lambda f: f"cleared-{f}(?n)")
    ops = [Operator.build("fix_a", parameters=("?n",), preconditions=("fault-a(?n)",),
                          add=(cleared("fault-a"),), delete=("fault-a(?n)",)),
           Operator.build("fix_b", parameters=("?n",), preconditions=("fault-b(?n)",),
                          add=(cleared("fault-b"),), delete=("fault-b(?n)",)),
           Operator.build("verify", parameters=("?n",),
                          preconditions=tuple({cleared("fault-a"), cleared("fault-b")}),
                          add=("service-restored(?n)",))]
    return Problem(name=f"compartilhado={shared}", operators=ops,
                   initial=make_state(["fault-a(N)", "fault-b(N)"]),
                   goal=make_state(["service-restored(N)"]),
                   objects={"node": ["N"]}, parameter_types={"?n": "node"})

for shared in (True, False):
    r = analyse(two_fault(shared), success=Predicate("service-restored", ("N",)),
                fault_literals=[Predicate("fault-a", ("N",)), Predicate("fault-b", ("N",))])
    print(r.render(LANG)); print()

Analise do grafo de planejamento: compartilhado=True
----------------------------------------------------
ponto fixo no nivel: 2
limite inferior do plano: 2
operadores mortos: nenhum
pontos de escolha:
  fault-cleared(N) <- fix_a, fix_b
violacoes do invariante de sucesso:
  service-restored(N) is not mutex with fault-a(N)
  service-restored(N) is not mutex with fault-b(N)

Analise do grafo de planejamento: compartilhado=False
-----------------------------------------------------
ponto fixo no nivel: 2
limite inferior do plano: 2
operadores mortos: nenhum
pontos de escolha: nenhum
violacoes do invariante de sucesso: nenhuma



---
## 3. Busca A\*

`f(n) = g(n) + h(n)`. A heurística é a distância em linha reta dividida pela
velocidade máxima do sistema — dividir pelo **máximo** é o que garante que a
estimativa nunca ultrapasse o custo real.

In [11]:
routing = RoutingProblem(topology, "NOC", "ER_03")
h = routing.heuristic()
print(f"{'estrategia':<24}{'saltos':>7}{'custo':>10}{'expandidos':>12}")
print("-" * 53)
for name, solver, informed in (("Largura", breadth_first, False),
                               ("Profundidade", depth_first, False),
                               ("Custo uniforme", uniform_cost, False),
                               ("Gulosa", greedy_best_first, True),
                               ("A*", astar, True)):
    r = solver(routing, h) if informed else solver(routing)
    print(f"{name:<24}{len(r.path)-1:>7}{r.cost:>10.2f}{r.expanded:>12}")

estrategia               saltos     custo  expandidos
-----------------------------------------------------
Largura                       4    453.51          15
Profundidade                  4    453.51           7
Custo uniforme                5    360.35          28
Gulosa                        5    360.35           7
A*                            5    360.35          23


**O caminho mais barato tem MAIS saltos.** A rota curta em saltos desce para a
malha de 900 MHz, lenta e com atraso de armazena-e-encaminha a cada repetidor; a
rota com um salto a mais permanece no LTE privativo, rápido. Contar saltos e
minimizar custo são objetivos diferentes, e aqui discordam.

In [12]:
best = astar(routing, h)
print(routing.explain_path(best.path, LANG))

salto                  meio                              qual.  custo(ms)  acum.(ms)
------------------------------------------------------------------------------------
NOC -> eNB_A           fibra optica                       0.98      60.00      60.00
eNB_A -> RELAY_1       LTE privativo                      0.90     128.73     188.73
RELAY_1 -> RELAY_5     LTE privativo                      0.90      86.33     275.05
RELAY_5 -> CPE_03      LTE privativo                      0.69      76.65     351.70
CPE_03 -> ER_03        ethernet local                     1.00       8.65     360.35


### 3.1 Verificação contra um oráculo independente

`uniform_cost` é literalmente `astar` com `h = 0`, então compará-los não é uma
verificação independente: um defeito comum corromperia os dois. Floyd–Warshall é
outro algoritmo, e por isso serve de referência externa. Ele respeita a mesma
regra de *stub*: um site é borda do cliente e nunca transporta tráfego alheio.

In [13]:
nodes = sorted(topology.nodes)
dist = {u: {v: (0.0 if u == v else math.inf) for v in nodes} for u in nodes}
for u in nodes:
    for v, cost in topology.successors(u):
        dist[u][v] = min(dist[u][v], cost)
for k in nodes:
    if topology.node(k).stub:      # customer edge: never an intermediate
        continue
    for i in nodes:
        if dist[i][k] == math.inf: continue
        for j in nodes:
            if dist[i][k] + dist[k][j] < dist[i][j]:
                dist[i][j] = dist[i][k] + dist[k][j]

mismatch = violations = pairs = 0
for s, g in itertools.permutations(nodes, 2):
    p = RoutingProblem(topology, s, g)
    r = astar(p, p.heuristic())
    if not r.found: continue
    pairs += 1
    if abs(r.cost - dist[s][g]) > 1e-6: mismatch += 1
    hh = p.heuristic()
    if hh(s) > dist[s][g] + 1e-9: violations += 1
print(f"pares verificados: {pairs}")
print(f"custos divergentes de Floyd-Warshall: {mismatch}")
print(f"heuristica inadmissivel em: {violations}")

pares verificados: 3540
custos divergentes de Floyd-Warshall: 0
heuristica inadmissivel em: 0


### 3.2 Esforço agregado — e um resultado que contraria a expectativa

In [14]:
def effort(name):
    t = load_topology(name)
    a = u = 0
    for s, g in itertools.permutations(sorted(t.nodes), 2):
        p = RoutingProblem(t, s, g)
        ra, ru = astar(p, p.heuristic()), uniform_cost(p)
        if ra.found:
            a += ra.expanded; u += ru.expanded
    return a, u

for name, label in (("simulated", "30 nos"), ("dual", "60 nos")):
    a, u = effort(name)
    print(f"{label:<8} A* {a:>7}   custo uniforme {u:>7}   reducao {100*(u-a)/u:>5.1f}%")

30 nos   A*   10310   custo uniforme   13920   reducao  25.9%


60 nos   A*   99963   custo uniforme  109740   reducao   8.9%


A redução **caiu** de 25,9% para 8,9% ao dobrar o número de nós. A vantagem da
heurística não cresceu com o grafo.

A explicação está na estrutura, não no tamanho: este cenário tem duas redes de
acesso quase paralelas, e a distância em linha reta informa pouco sobre qual meio
é mais barato quando os meios têm velocidades diferentes. São duas medições em
dois grafos — e continuam sendo apenas duas medições.

### 3.3 Perda de um meio e comutação para o outro

In [15]:
degraded = load_topology("dual")
degraded.disable_link("NOC", "eNB_A")        # remove todo o acesso pLTE do nucleo
p = RoutingProblem(degraded, "NOC", "ER_03")
r = astar(p, p.heuristic())
print("apos a falha:", " -> ".join(r.path))
print(f"custo {r.cost:.2f} ms  (antes: {best.cost:.2f} ms pelo pLTE)")

apos a falha: NOC -> SAF_01 -> SAF_02 -> RM_03 -> ER_03
custo 453.51 ms  (antes: 360.35 ms pelo pLTE)


O site **não fica isolado**: comuta para a malha de 900 MHz e o preço aparece no
custo. É exatamente isso que o duplo acesso compra.

E quando não há rota, a resposta correta é ausência de rota — não um plano que
pressupõe desvio disponível.

In [16]:
isolated = RoutingProblem(topology, "NOC", "ER_03", avoid=("CPE_03", "RM_03"))
print("rota encontrada:", astar(isolated, isolated.heuristic()).found)

rota encontrada: False


---
## Referências

- Fikes, R. E., & Nilsson, N. J. (1971). STRIPS. *Artificial Intelligence*, 2(3–4).
- Blum, A. L., & Furst, M. L. (1997). Fast planning through planning graph analysis.
  *Artificial Intelligence*, 90(1–2).
- Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A formal basis for the heuristic
  determination of minimum cost paths. *IEEE Trans. SSC*, 4(2).
- Shortliffe, E. H., & Buchanan, B. G. (1975). A model of inexact reasoning in
  medicine. *Mathematical Biosciences*, 23(3–4).